# Build multilingual lexicons

This notebook generates `lexicon_{language_code}.txt` for each language returned by `available_languages(wordlist="best")`.

Output directory:
`./data/lexicon/`

Each output file has 3 columns:
- `word`
- `length`
- `frequency_rank`

Pipeline:
1. Get candidate words with `top_n_list(lang, MAX_WORDS_PER_LANGUAGE, wordlist="best")`.
2. Remove noisy entries (numbers, emoji, and non-word-like tokens).
3. Apply script-based language filtering to reduce cross-language leakage
   (e.g., Latin/Kana/Hangul tokens in `zh` lexicon).
4. Compute word lengths and apply rule:
   - if language average length `> 3`, set `length = max(length, 3)`;
   - otherwise keep original length.
5. Compute frequency score as `-log2(probability)` and rank by ascending score (most frequent = rank 1).
6. Save to `lexicon_{language_code}.txt`.

CJK note:
- If CJK tokenizer dependencies are unavailable, the notebook can fall back to `get_frequency_dict` for candidate words/frequencies for that language.

In [1]:
import csv
import math
import unicodedata
from pathlib import Path

import regex
from wordfreq import available_languages, get_frequency_dict, top_n_list, word_frequency

In [2]:
# Max number of words per language (same cap idea as before).
MAX_WORDS_PER_LANGUAGE = 100000

OUTPUT_DIR = Path("/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = sorted(available_languages(wordlist="best").keys())
print(f"Supported language codes: {len(LANGUAGES)}")
print(LANGUAGES)

Supported language codes: 42
['ar', 'bg', 'bn', 'ca', 'cs', 'da', 'de', 'el', 'en', 'es', 'fa', 'fi', 'fil', 'fr', 'he', 'hi', 'hu', 'id', 'is', 'it', 'ja', 'ko', 'lt', 'lv', 'mk', 'ms', 'nb', 'nl', 'pl', 'pt', 'ro', 'ru', 'sh', 'sk', 'sl', 'sv', 'ta', 'tr', 'uk', 'ur', 'vi', 'zh']


In [3]:
ALLOWED_CONNECTORS = {"'", "’", "ʼ", "-", "‐", "‑", "·"}
NUMBER_RE = regex.compile(r"\p{Number}")
EMOJI_RE = regex.compile(r"\p{Extended_Pictographic}")
LETTER_RE = regex.compile(r"\p{Letter}")

HAS_LATIN = regex.compile(r"\p{Script=Latin}")
HAS_HAN = regex.compile(r"\p{Script=Han}")
HAS_HIRAGANA = regex.compile(r"\p{Script=Hiragana}")
HAS_KATAKANA = regex.compile(r"\p{Script=Katakana}")
HAS_HANGUL = regex.compile(r"\p{Script=Hangul}")
HAS_CYRILLIC = regex.compile(r"\p{Script=Cyrillic}")
HAS_ARABIC = regex.compile(r"\p{Script=Arabic}")
HAS_HEBREW = regex.compile(r"\p{Script=Hebrew}")
HAS_GREEK = regex.compile(r"\p{Script=Greek}")
HAS_DEVANAGARI = regex.compile(r"\p{Script=Devanagari}")
HAS_BENGALI = regex.compile(r"\p{Script=Bengali}")
HAS_TAMIL = regex.compile(r"\p{Script=Tamil}")

LATIN_LANGS = {
    "ca", "cs", "da", "de", "en", "es", "fi", "fil", "fr", "hu", "id", "is",
    "it", "lt", "lv", "ms", "nb", "nl", "pl", "pt", "ro", "sh", "sk", "sl", "sv",
    "tr", "vi",
}


def compute_frequency(word, language, freq_lookup=None):
    """
    Compute lexical frequency as negative log2(word_probability).

    Prefer direct lookup from get_frequency_dict() so we don't depend on
    language tokenizers (e.g., MeCab for ja/ko) during ranking.
    """
    freq = None

    if freq_lookup is not None:
        freq = freq_lookup.get(word)

    if freq is None:
        try:
            freq = word_frequency(word, language)
        except ModuleNotFoundError:
            return float("inf")

    if freq <= 0:
        return float("inf")
    return round(-math.log2(freq), 2)


def belongs_to_target_language(word, lang):
    """
    Heuristic script-based language identity filter.
    This removes obvious cross-language leakage in lexicons (e.g., English in zh).
    """
    code = str(lang).strip().lower()

    if code == "zh":
        return bool(HAS_HAN.search(word)) and not (
            HAS_LATIN.search(word) or HAS_HIRAGANA.search(word) or HAS_KATAKANA.search(word) or HAS_HANGUL.search(word)
        )

    if code == "ja":
        return bool(HAS_HAN.search(word) or HAS_HIRAGANA.search(word) or HAS_KATAKANA.search(word)) and not HAS_HANGUL.search(word)

    if code == "ko":
        return bool(HAS_HANGUL.search(word)) and not (
            HAS_LATIN.search(word) or HAS_HIRAGANA.search(word) or HAS_KATAKANA.search(word)
        )

    if code in {"ru", "uk", "bg", "mk"}:
        return bool(HAS_CYRILLIC.search(word))

    if code in {"ar", "fa", "ur"}:
        return bool(HAS_ARABIC.search(word))

    if code == "he":
        return bool(HAS_HEBREW.search(word))

    if code == "el":
        return bool(HAS_GREEK.search(word))

    if code == "hi":
        return bool(HAS_DEVANAGARI.search(word))

    if code == "bn":
        return bool(HAS_BENGALI.search(word))

    if code == "ta":
        return bool(HAS_TAMIL.search(word))

    if code in LATIN_LANGS:
        return bool(HAS_LATIN.search(word)) and not (
            HAS_HAN.search(word) or HAS_HIRAGANA.search(word) or HAS_KATAKANA.search(word) or HAS_HANGUL.search(word)
        )

    # Unknown language code: do not filter by script.
    return True


def is_valid_word(word, lang):
    """
    Keep entries that look like real words and drop numeric/symbolic noise.
    Rules:
    - must contain at least one letter
    - no numbers
    - no emoji
    - only letters/marks and a small set of in-word connectors
    - should belong to target language script profile
    """
    word = word.strip()
    if not word:
        return False

    if NUMBER_RE.search(word):
        return False

    if EMOJI_RE.search(word):
        return False

    if not LETTER_RE.search(word):
        return False

    for ch in word:
        if ch in ALLOWED_CONNECTORS:
            continue
        category = unicodedata.category(ch)
        if category[0] in {"L", "M"}:
            continue
        return False

    if not belongs_to_target_language(word, lang):
        return False

    return True


def adjusted_lengths(words):
    """
    If avg length > 3 for a language, clamp each length to min 3.
    Else keep original lengths.
    """
    if not words:
        return [], 0.0, False

    raw_lengths = [len(w) for w in words]
    avg_len = sum(raw_lengths) / len(raw_lengths)
    apply_min_three = avg_len > 3

    if apply_min_three:
        lengths = [max(3, l) for l in raw_lengths]
    else:
        lengths = raw_lengths

    return lengths, avg_len, apply_min_three

In [4]:
def _get_words_for_language(lang, max_words):
    """
    Prefer top_n_list(..., wordlist='best').
    If CJK tokenizer deps are missing (e.g., MeCab), fall back to frequency dict.
    """
    try:
        words = top_n_list(lang, max_words, wordlist="best")
        return words, "top_n_list"
    except ModuleNotFoundError:
        freq_lookup = get_frequency_dict(lang, wordlist="best")
        words = sorted(freq_lookup, key=freq_lookup.get, reverse=True)[:max_words]
        return words, "get_frequency_dict_fallback"


def create_lexicon_for_language(lang, max_words=MAX_WORDS_PER_LANGUAGE):
    words, source = _get_words_for_language(lang, max_words)

    filtered_words = [w for w in words if is_valid_word(w, lang)]
    filtered_words = list(dict.fromkeys(filtered_words))

    lengths, avg_len, apply_min_three = adjusted_lengths(filtered_words)

    freq_lookup = get_frequency_dict(lang, wordlist="best")
    data = [
        (w, l, compute_frequency(w, lang, freq_lookup=freq_lookup))
        for w, l in zip(filtered_words, lengths)
    ]
    data = [row for row in data if math.isfinite(row[2])]

    data.sort(key=lambda x: x[2])

    data_with_rank = [
        (w, l, rank + 1)
        for rank, (w, l, _) in enumerate(data)
    ]

    output_path = OUTPUT_DIR / f"lexicon_{lang}.txt"
    with output_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow(["word", "length", "frequency_rank"])
        writer.writerows(data_with_rank)

    return {
        "lang": lang,
        "input_words": len(words),
        "kept_words": len(filtered_words),
        "ranked_words": len(data_with_rank),
        "avg_len": round(avg_len, 3),
        "min_len_3_applied": apply_min_three,
        "source": source,
        "output": str(output_path),
    }


results = []
errors = []

for lang in LANGUAGES:
    try:
        result = create_lexicon_for_language(lang)
        results.append(result)
        print(
            f"[{lang}] kept {result['kept_words']}/{result['input_words']} "
            f"ranked={result['ranked_words']} source={result['source']} "
            f"(avg_len={result['avg_len']}, min3={result['min_len_3_applied']})"
        )
    except Exception as exc:
        errors.append((lang, str(exc)))
        print(f"[{lang}] ERROR: {exc}")

print("\nFinished.")
print(f"Succeeded: {len(results)}")
print(f"Failed: {len(errors)}")
if errors:
    print(errors)

[ar] kept 97645/100000 ranked=97645 source=top_n_list (avg_len=5.711, min3=True)
[bg] kept 35849/37271 ranked=35849 source=top_n_list (avg_len=7.638, min3=True)
[bn] kept 93251/100000 ranked=93251 source=top_n_list (avg_len=7.295, min3=True)
[ca] kept 98986/100000 ranked=98986 source=top_n_list (avg_len=7.668, min3=True)
[cs] kept 99566/100000 ranked=99566 source=top_n_list (avg_len=7.551, min3=True)
[da] kept 29250/29399 ranked=29250 source=top_n_list (avg_len=7.35, min3=True)
[de] kept 99224/100000 ranked=99224 source=top_n_list (avg_len=9.113, min3=True)
[el] kept 43102/46808 ranked=43102 source=top_n_list (avg_len=8.031, min3=True)
[en] kept 98176/100000 ranked=98176 source=top_n_list (avg_len=7.212, min3=True)
[es] kept 99078/100000 ranked=99078 source=top_n_list (avg_len=7.852, min3=True)
[fa] kept 27689/31317 ranked=27689 source=top_n_list (avg_len=5.232, min3=True)
[fi] kept 99285/100000 ranked=99285 source=top_n_list (avg_len=9.07, min3=True)
[fil] kept 30071/30226 ranked=3007

In [5]:
# Quick summary table (optional)
try:
    import pandas as pd

    summary_df = pd.DataFrame(results).sort_values("lang")
    display(summary_df)
except Exception:
    print("Install pandas if you want a tabular summary.")
    print(results[:5])

,lang,input_words,kept_words,ranked_words,avg_len,min_len_3_applied,source,output
0,ar,100000,97645,97645,5.711,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
1,bg,37271,35849,35849,7.638,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
2,bn,100000,93251,93251,7.295,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
3,ca,100000,98986,98986,7.668,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
4,cs,100000,99566,99566,7.551,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
5,da,29399,29250,29250,7.350,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
6,de,100000,99224,99224,9.113,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
7,el,46808,43102,43102,8.031,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
8,en,100000,98176,98176,7.212,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
9,es,100000,99078,99078,7.852,True,top_n_list,/swdata/yin/Cui/LLM-MAZE/llmmaze/data/lexicon/...
